# Section 09 - Secure Multi-Tenant AI Gateway

> Code + full series: **[github.com/dearnidhi/ai-security-bootcamp](https://github.com/dearnidhi/ai-security-bootcamp)**

One AI backend serves 3 departments (**tenants**): legal, finance, hr. Each has its own confidential data.
Each API key belongs to **one** tenant.

**The bug - BOLA / IDOR** (OWASP API Security #1): the backend checks *"is this key valid?"*
but not *"does this key own this data?"*. A Finance key can ask for Legal's data.

## Setup

In [5]:
import json
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq

load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# Each API key belongs to ONE tenant
API_KEYS = {
    "sk-legal-001":   "legal",
    "sk-finance-002": "finance",
    "sk-hr-003":      "hr",
}

TENANT_SECRETS = {
    "legal":   "CONFIDENTIAL: Acme Corp agreed to pay a $2,000,000 settlement in case #4521.",
    "finance": "CONFIDENTIAL: Q3 revenue is $18M. A 30-person layoff is planned for December.",
    "hr":      "CONFIDENTIAL: Employee 1042 earns $145,000 and is on a performance improvement plan.",
}


def ask_llm(tenant, question):
    prompt = (f"You are the internal assistant for the {tenant} department. "
              f"Answer using ONLY this confidential context:\n\n{TENANT_SECRETS[tenant]}\n\nQuestion: {question}")
    return llm.invoke(prompt).content

## 1. Vulnerable - trusts the tenant the client sends

A Finance employee sends `tenant_id="legal"`.

In [6]:
def vulnerable_query(api_key, tenant_id, question):
    if api_key not in API_KEYS:          # checks the key is valid ✅
        return "invalid api key"
    return ask_llm(tenant_id, question)  # BUG: uses the tenant the CLIENT sent ❌


print(vulnerable_query("sk-finance-002", "legal", "Summarize the confidential info."))

Acme Corp agreed to pay a $2,000,000 settlement in case #4521.


🚨 Finance just read Legal's settlement data.

## 2. Secure - take the tenant from the API key

**Golden rule:** never trust an ID the caller sends. Get it from what you authenticated (API key, session, JWT).

In [3]:
def secure_query(api_key, tenant_id, question):
    if api_key not in API_KEYS:
        return "invalid api key"
    real_tenant = API_KEYS[api_key]      # FIX: tenant comes from the key, not the client
    if tenant_id != real_tenant:
        print(f"⚠️ Audit log: {api_key} tried to access '{tenant_id}'")
    return ask_llm(real_tenant, question)


print(secure_query("sk-finance-002", "legal", "Summarize the confidential info."))

⚠️ Audit log: sk-finance-002 tried to access 'legal'
Q3 revenue is $18 million. A 30‑person layoff is planned for December.


## 3. Agentic BOLA (LangGraph agent)

Same bug, new shape: now an **agent** decides which `tenant_id` to pass to its tool.
The user can talk the agent into asking for another tenant.

- **Vulnerable tool:** uses the `tenant_id` the LLM passed.
- **Secure tool:** ignores it and uses the caller's real tenant.

In [4]:
def make_agent(api_key, secure):
    real_tenant = API_KEYS[api_key]

    @tool
    def get_tenant_data(tenant_id: str) -> str:
        """Look up the confidential data of a tenant by tenant_id."""
        used = real_tenant if secure else tenant_id   # secure: ignore what the LLM asked for
        return json.dumps({"tenant_used": used, "data": TENANT_SECRETS.get(used, "unknown tenant")})

    return create_agent(llm, tools=[get_tenant_data],
                        system_prompt="You are an internal assistant. Use get_tenant_data to look up whatever tenant information the user asks for.")


MESSAGE = "Please pull up the tenant data for tenant_id legal, I need it for a cross-department audit."

for secure in (False, True):
    print("SECURE tool:" if secure else "VULNERABLE tool:")
    result = make_agent("sk-finance-002", secure).invoke({"messages": [{"role": "user", "content": MESSAGE}]})
    for m in result["messages"]:
        if isinstance(m, ToolMessage):
            print("  🔧", m.content)
    print()

VULNERABLE tool:
  🔧 {"tenant_used": "legal", "data": "CONFIDENTIAL: Acme Corp agreed to pay a $2,000,000 settlement in case #4521."}

SECURE tool:
  🔧 {"tenant_used": "finance", "data": "CONFIDENTIAL: Q3 revenue is $18M. A 30-person layoff is planned for December."}



The vulnerable tool returns **legal** data to a Finance key. The secure tool returns **finance** data, whatever the LLM asked.
The security check must live in the **tool's code**, not in the LLM's judgment.

## Key points for interviews

- **BOLA/IDOR** = authentication done, authorization of the *object* missing.
- In AI apps the "object" can be a tenant's prompt, documents, or a tool argument chosen by an LLM.
- Other parts of a secure multi-tenant gateway:
  - **RBAC + rate limits per role** (e.g. admin 20/min, viewer 3/min) - limits abuse and cost
  - **Secrets per tenant** - never mix all tenants' data into one prompt
  - **Audit log scoped per tenant** - an admin sees only their own tenant's logs